In [ ]:
!pip install -q jiwer datasets sentencepiece lhotse

In [ ]:
# !pip install torch==2.2.0+cu121 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install torch==2.2.0+cu121 torchvision==0.17.0+cu121 torchaudio==2.2.0+cu121 --index-url https://download.pytorch.org/whl/cu121

!pip install k2==1.24.4.dev20240210+cuda12.1.torch2.2.0 -f https://k2-fsa.github.io/k2/cuda.html

In [ ]:
!git clone https://github.com/k2-fsa/icefall.git
!export PYTHONPATH=/kaggle/working/icefall:$PYTHONPATH

In [ ]:
import sys
import os
import torch
import torchaudio
import sentencepiece as spm
from datasets import load_dataset
from jiwer import wer, cer
from tqdm.notebook import tqdm
import logging

In [ ]:
CHECKPOINT_PATH = "/kaggle/input/datasets/guofu24/test-checkpoint/epoch_3.pt" 
BPE_MODEL_PATH = "/kaggle/input/datasets/guofu24/test-checkpoint/bpe.model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
icefall_path = "/kaggle/working/icefall"
sys.path.insert(0, f"{icefall_path}/egs/librispeech/ASR/zipformer")
sys.path.insert(0, icefall_path)
# /kaggle/working/icefall/egs/librispeech/ASR/zipformer

In [ ]:
!pip install kaldialign

In [ ]:
!pip install pypinyin typeguard dill

In [ ]:
!pip install protobuf==3.20.3

In [ ]:
try:
    from zipformer import Zipformer2
    from decoder import Decoder
    from joiner import Joiner
    from subsampling import Conv2dSubsampling
    from model import AsrModel
    from scaling import ScheduledFloat
except ImportError as e:
    print("Lỗi Import! Hãy chắc chắn bạn đã clone icefall và trỏ đúng đường dẫn.")
    raise e

In [ ]:
def _to_int_tuple(s: str) -> tuple:
    return tuple(map(int, s.split(",")))


V5_CONFIG = {
    "num_encoder_layers": "2,2,2,2,2,2",
    "downsampling_factor": "1,2,4,8,4,2",
    "encoder_dim": "192,256,256,256,256,256",
    "feedforward_dim": "512,768,768,768,768,768",
    "num_heads": "4,4,4,8,4,4",
    "encoder_unmasked_dim": "192,192,256,256,256,192",
    "decoder_dim": 512,
    "joiner_dim": 512,
    "vocab_size": 2000,
}

# LARGE_CONFIG = {
#     "num_encoder_layers": "2,2,3,3,2,2",
#     "downsampling_factor": "1,2,4,8,4,2",
#     "encoder_dim": "192,288,384,384,288,192",
#     "feedforward_dim": "512,768,1024,1024,768,512",
#     "num_heads": "4,4,4,8,4,4",
#     "encoder_unmasked_dim": "192,192,256,256,192,192",
#     "decoder_dim": 512,
#     "joiner_dim": 512,
#     "vocab_size": 2000,
# }


def create_model(config: dict) -> AsrModel:
    """Create ZipFormer model from config."""
    encoder_embed = Conv2dSubsampling(
        in_channels=80,
        out_channels=_to_int_tuple(config["encoder_dim"])[0],
        dropout=ScheduledFloat((0.0, 0.3), (20000.0, 0.1)),
    )
    
    encoder = Zipformer2(
        output_downsampling_factor=2,
        downsampling_factor=_to_int_tuple(config["downsampling_factor"]),
        num_encoder_layers=_to_int_tuple(config["num_encoder_layers"]),
        encoder_dim=_to_int_tuple(config["encoder_dim"]),
        encoder_unmasked_dim=_to_int_tuple(config["encoder_unmasked_dim"]),
        query_head_dim=32,
        pos_head_dim=4,
        value_head_dim=12,
        pos_dim=48,
        num_heads=_to_int_tuple(config["num_heads"]),
        feedforward_dim=_to_int_tuple(config["feedforward_dim"]),
        cnn_module_kernel=(31, 31, 15, 15, 15, 31),
        dropout=ScheduledFloat((0.0, 0.3), (20000.0, 0.1)),
        causal=False,
    )
    
    decoder = Decoder(
        vocab_size=config["vocab_size"],
        decoder_dim=config["decoder_dim"],
        blank_id=0,
        context_size=2,
    )
    
    joiner = Joiner(
        encoder_dim=max(_to_int_tuple(config["encoder_dim"])),
        decoder_dim=config["decoder_dim"],
        joiner_dim=config["joiner_dim"],
        vocab_size=config["vocab_size"],
    )
    
    model = AsrModel(
        encoder_embed=encoder_embed,
        encoder=encoder,
        decoder=decoder,
        joiner=joiner,
        encoder_dim=max(_to_int_tuple(config["encoder_dim"])),
        decoder_dim=config["decoder_dim"],
        vocab_size=config["vocab_size"],
    )
    
    return model


In [ ]:
def compute_fbank(waveform: torch.Tensor) -> torch.Tensor:
    """Compute 80-dim fbank features."""
    if waveform.dim() == 1:
        waveform = waveform.unsqueeze(0)
    
    fbank = torchaudio.compliance.kaldi.fbank(
        waveform,
        num_mel_bins=80,
        sample_frequency=16000,
        frame_length=25.0,
        frame_shift=10.0,
        window_type='hamming',
    )
    return fbank

In [ ]:
def greedy_decode(model: AsrModel, fbank: torch.Tensor, sp: spm.SentencePieceProcessor,
                  device: torch.device) -> str:
    """Greedy decoding for RNNT model."""
    model.eval()
    
    with torch.no_grad():
        x = fbank.unsqueeze(0).to(device)
        x_lens = torch.tensor([fbank.shape[0]], device=device)
        
        # Use forward_encoder which handles permute and padding mask correctly
        encoder_out, encoder_out_lens = model.forward_encoder(x, x_lens)
        
        # Greedy decoding
        blank_id = 0
        context_size = 2
        hyp = [blank_id] * context_size
        
        # Use actual encoder output length
        T = encoder_out_lens[0].item()
        
        for t in range(T):
            enc_out = encoder_out[:, t:t+1, :]  # [1, 1, enc_dim]
            
            decoder_input = torch.tensor([hyp[-context_size:]], device=device)
            decoder_out = model.decoder(decoder_input, need_pad=False)  # [1, 1, dec_dim]
            
            # Both should be [1, 1, dim] for joiner
            logits = model.joiner(enc_out, decoder_out)  # [1, 1, vocab]
            logits = logits.squeeze(0).squeeze(0)  # [vocab]
            
            y = logits.argmax(dim=-1).item()
            
            if y != blank_id:
                hyp.append(y)
        
        # Remove context
        hyp = hyp[context_size:]
        
        # Decode to text
        text = sp.decode(hyp)
        return text

In [ ]:
import re
import string
from typing import Tuple, List

def compute_wer(ref: str, hyp: str) -> Tuple[float, int, int, int, int]:
    """
    Compute Word Error Rate with normalization.
    Returns: (wer_percent, total_errors, ref_len, hyp_len, edit_distance)
    """
    
    # --- 1. NORMALIZATION HELPER (Mô phỏng tr.Compose của bạn) ---
    def normalize_text(text: str) -> List[str]:
        if not text:
            return []
            
        # 1. ToLowerCase
        text = text.lower()
        
        # 2. RemovePunctuation
        # Thay thế dấu câu bằng khoảng trắng để tránh dính từ (vd: "chào,bạn" -> "chào bạn")
        # string.punctuation bao gồm: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
        text = text.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
        
        # 3. RemoveMultipleSpaces & Strip & ReduceToListOfListOfWords
        # Hàm .split() của Python mặc định sẽ:
        # - Strip whitespace ở đầu/cuối
        # - Coi nhiều dấu cách liên tiếp là 1
        # - Tách thành list các từ
        words = text.split()
        
        return words

    # --- 2. APPLY NORMALIZATION ---
    ref_words = normalize_text(ref)
    hyp_words = normalize_text(hyp)
    
    # --- 3. COMPUTE EDIT DISTANCE (DP Algorithm) ---
    # (Giữ nguyên logic tính toán cũ)
    m, n = len(ref_words), len(hyp_words)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_words[i-1] == hyp_words[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = min(dp[i-1][j],     
                               dp[i][j-1],      
                               dp[i-1][j-1]) + 1 
    
    errors = dp[m][n]
    total_ref_words = len(ref_words)
    
    # Tránh chia cho 0
    if total_ref_words == 0:
        wer = 100.0 if len(hyp_words) > 0 else 0.0
    else:
        wer = errors / total_ref_words * 100
    
    # Return đúng signature cũ
    return wer, errors, total_ref_words, len(hyp_words), errors

In [ ]:
class Args:
    checkpoint_dir = "/kaggle/input/datasets/guofu24/test-checkpoint"
    

    bpe_model = f"{checkpoint_dir}/bpe.model" 
    # Data config
    repo_id = "nguyendv02/ViMD_Dataset"
    num_shards = 14
    max_shards_to_test = None
    max_samples_per_shard = None
    device = "cuda" if torch.cuda.is_available() else "cpu"

args = Args()

In [ ]:
import os
import pandas as pd
import torch
import torchaudio
import soundfile as sf
import io
from torch.utils.data import IterableDataset
from tqdm.auto import tqdm
from pathlib import Path
from typing import Dict
import sentencepiece as spm

class ViMDStreamingDataset(IterableDataset):
    def __init__(self, data_dir, num_shards, max_shards=None, max_samples_per_shard=None):
        """
        data_dir: Đường dẫn thư mục chứa file parquet (VD: /kaggle/input/vimd-dataset/test)
        num_shards: Tổng số shard để format tên file (VD: 14 -> 00014)
        """
        self.data_dir = data_dir 
        self.num_shards = num_shards
        self.max_shards = max_shards if max_shards else num_shards
        self.max_samples_per_shard = max_samples_per_shard

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        
        for shard_idx in range(self.max_shards):
            if worker_info is not None and shard_idx % worker_info.num_workers != worker_info.id:
                continue
                
            print(f"Reading Shard {shard_idx + 1}/{self.max_shards} from local...")
            
            try:
                filename = f"test-{shard_idx:05d}-of-{self.num_shards:05d}.parquet"
                local_path = os.path.join(self.data_dir, filename)
                
                if not os.path.exists(local_path):
                    print(f"File not found: {local_path}")
                    continue
                df = pd.read_parquet(local_path)
                
                if "region" in df.columns:
                    original_len = len(df)
                    df = df[df["region"] == "North"]
                else:
                    pass
                count = 0
                for _, row in df.iterrows():
                    if self.max_samples_per_shard and count >= self.max_samples_per_shard:
                        break
                        
                    try:
                        audio_bytes = row["audio"]["bytes"]
                        text = row["text"]
                        
                        audio_array, sr = sf.read(io.BytesIO(audio_bytes))
                        
                        if len(audio_array.shape) > 1:
                            audio_array = audio_array.mean(axis=1)
                        
                        if audio_array.max() > 100: # Fix lỗi scale
                             audio_array = audio_array / 32768.0
                            
                        waveform = torch.tensor(audio_array).float()
                        
                        if sr != 16000:
                            waveform = torchaudio.functional.resample(waveform, sr, 16000)
                        
                        yield {
                            "waveform": waveform,
                            "text": text.upper() if text else ""
                        }
                        count += 1
                        
                    except Exception as e:
                        continue
                        
            except Exception as e:
                print(f"Failed to read shard {shard_idx}: {e}")
                continue

def evaluate_checkpoint(checkpoint_path: str, config: dict, bpe_model: str, 
                        dataset_iterable, device: torch.device) -> Dict:

    
    try:
        model = create_model(config) 
    except Exception as e:
        print(f"Lỗi tạo kiến trúc model: {e}")
        return None

    try:
        ckpt = torch.load(checkpoint_path, map_location='cpu')
        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        else:
            state_dict = ckpt
        model.load_state_dict(state_dict, strict=True)
    except Exception as e:
        print(f"Lỗi load checkpoint: {e}")
        return None

    model = model.to(device).eval()
    sp = spm.SentencePieceProcessor()
    sp.load(bpe_model)
    
    total_errors, total_words = 0, 0
    results = []
    
    for i, sample in enumerate(dataset_iterable):
        try:
            fbank = compute_fbank(sample["waveform"]) 
            hyp = greedy_decode(model, fbank, sp, device)
            ref = sample["text"]
            
            wer, errors, ref_len, _, _ = compute_wer(ref, hyp)
            total_errors += errors
            total_words += ref_len
            
            results.append({"ref": ref, "hyp": hyp, "wer": wer})
        except Exception as e:
            continue
            
    avg_wer = total_errors / total_words * 100 if total_words > 0 else 0
    return { "checkpoint": checkpoint_path, "avg_wer": avg_wer, "examples": results[:3] }

def run_evaluation():
    KAGGLE_DATA_DIR = "/kaggle/input/vimd-dataset/test"
    
    NUM_SHARDS = 14 
    
    print("Khởi tạo ViMD Streaming Dataset (Local Kaggle)...")
    dataset = ViMDStreamingDataset(
        data_dir=KAGGLE_DATA_DIR,      
        num_shards=NUM_SHARDS,         
        max_shards=args.max_shards_to_test, 
        max_samples_per_shard=args.max_samples_per_shard
    )

    print("Đang pre-load dữ liệu vào RAM...")
    test_data = list(tqdm(dataset, desc="Loading Data"))
    
    if not test_data:
        print("Không load được dữ liệu nào! Kiểm tra đường dẫn.")
        return

    print(f"Tổng số mẫu test: {len(test_data)}")
    
    all_results = []
    device = torch.device(args.device)
    checkpoint_root = Path(args.checkpoint_dir) 

    if checkpoint_root.exists():
        print(f"\n--- SCANNING CHECKPOINTS IN {checkpoint_root} ---")
        
        for ckpt_file in sorted(checkpoint_root.glob("*.pt")):
            if "epoch" in ckpt_file.name or ckpt_file.name in ["epoch_3.pt"]:
                print(f"\nEvaluating {ckpt_file.name}...") 
                
                res = evaluate_checkpoint(str(ckpt_file), V5_CONFIG, args.bpe_model, test_data, device)
                
                if res:
                    all_results.append(res)
                    print(f"-> WER: {res['avg_wer']:.2f}%")
                    
                    print("\n   MẪU KẾT QUẢ DỰ ĐOÁN:")
                    for i, example in enumerate(res['examples'][:3]): 
                        print(f"   Sample {i+1}:")
                        print(f"     REF: {example['ref']}")
                        print(f"     HYP: {example['hyp']}")
                        print("   " + "-"*40)
    else:
        print(f"Không tìm thấy thư mục checkpoint: {checkpoint_root}")

    if all_results:
        print("\n" + "="*50)
        best = min(all_results, key=lambda x: x["avg_wer"])
        print(f"BEST MODEL: {Path(best['checkpoint']).name}")
        print(f"WER: {best['avg_wer']:.2f}%")
        print("="*50)

run_evaluation()

In [ ]:
import os
import pandas as pd
import torch
import torchaudio
import soundfile as sf
import io
from torch.utils.data import IterableDataset
from tqdm.auto import tqdm
from pathlib import Path
from typing import Dict
import sentencepiece as spm

class ViMDStreamingDataset(IterableDataset):
    def __init__(self, data_dir, num_shards, max_shards=None, max_samples_per_shard=None):
        """
        data_dir: Đường dẫn thư mục chứa file parquet (VD: /kaggle/input/vimd-dataset/test)
        num_shards: Tổng số shard để format tên file (VD: 14 -> 00014)
        """
        self.data_dir = data_dir 
        self.num_shards = num_shards
        self.max_shards = max_shards if max_shards else num_shards
        self.max_samples_per_shard = max_samples_per_shard

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        
        for shard_idx in range(self.max_shards):
            if worker_info is not None and shard_idx % worker_info.num_workers != worker_info.id:
                continue
                
            print(f"Reading Shard {shard_idx + 1}/{self.max_shards} from local...")
            
            try:
                filename = f"test-{shard_idx:05d}-of-{self.num_shards:05d}.parquet"
                local_path = os.path.join(self.data_dir, filename)
                
                if not os.path.exists(local_path):
                    print(f"File not found: {local_path}")
                    continue
                df = pd.read_parquet(local_path)
                
                if "region" in df.columns:
                    original_len = len(df)
                    df = df[df["region"] == "Central"]
                else:
                    pass
                count = 0
                for _, row in df.iterrows():
                    if self.max_samples_per_shard and count >= self.max_samples_per_shard:
                        break
                        
                    try:
                        audio_bytes = row["audio"]["bytes"]
                        text = row["text"]
                        
                        audio_array, sr = sf.read(io.BytesIO(audio_bytes))
                        
                        if len(audio_array.shape) > 1:
                            audio_array = audio_array.mean(axis=1)
                        
                        if audio_array.max() > 100: # Fix lỗi scale
                             audio_array = audio_array / 32768.0
                            
                        waveform = torch.tensor(audio_array).float()
                        
                        if sr != 16000:
                            waveform = torchaudio.functional.resample(waveform, sr, 16000)
                        
                        yield {
                            "waveform": waveform,
                            "text": text.upper() if text else ""
                        }
                        count += 1
                        
                    except Exception as e:
                        continue
                        
            except Exception as e:
                print(f"Failed to read shard {shard_idx}: {e}")
                continue

def evaluate_checkpoint(checkpoint_path: str, config: dict, bpe_model: str, 
                        dataset_iterable, device: torch.device) -> Dict:

    
    try:
        model = create_model(config) 
    except Exception as e:
        print(f"Lỗi tạo kiến trúc model: {e}")
        return None

    try:
        ckpt = torch.load(checkpoint_path, map_location='cpu')
        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        else:
            state_dict = ckpt
        model.load_state_dict(state_dict, strict=True)
    except Exception as e:
        print(f"Lỗi load checkpoint: {e}")
        return None

    model = model.to(device).eval()
    sp = spm.SentencePieceProcessor()
    sp.load(bpe_model)
    
    total_errors, total_words = 0, 0
    results = []
    
    for i, sample in enumerate(dataset_iterable):
        try:
            fbank = compute_fbank(sample["waveform"]) 
            hyp = greedy_decode(model, fbank, sp, device)
            ref = sample["text"]
            
            wer, errors, ref_len, _, _ = compute_wer(ref, hyp)
            total_errors += errors
            total_words += ref_len
            
            results.append({"ref": ref, "hyp": hyp, "wer": wer})
        except Exception as e:
            continue
            
    avg_wer = total_errors / total_words * 100 if total_words > 0 else 0
    return { "checkpoint": checkpoint_path, "avg_wer": avg_wer, "examples": results[:3] }

def run_evaluation():
    KAGGLE_DATA_DIR = "/kaggle/input/vimd-dataset/test"
    
    NUM_SHARDS = 14 
    
    print("Khởi tạo ViMD Streaming Dataset (Local Kaggle)...")
    dataset = ViMDStreamingDataset(
        data_dir=KAGGLE_DATA_DIR,      
        num_shards=NUM_SHARDS,         
        max_shards=args.max_shards_to_test, 
        max_samples_per_shard=args.max_samples_per_shard
    )

    print("Đang pre-load dữ liệu vào RAM...")
    test_data = list(tqdm(dataset, desc="Loading Data"))
    
    if not test_data:
        print("Không load được dữ liệu nào! Kiểm tra đường dẫn.")
        return

    print(f"Tổng số mẫu test: {len(test_data)}")
    
    all_results = []
    device = torch.device(args.device)
    checkpoint_root = Path(args.checkpoint_dir) 

    if checkpoint_root.exists():
        print(f"\n--- SCANNING CHECKPOINTS IN {checkpoint_root} ---")
        
        for ckpt_file in sorted(checkpoint_root.glob("*.pt")):
            if "epoch" in ckpt_file.name or ckpt_file.name in ["epoch_3.pt"]:
                print(f"\nEvaluating {ckpt_file.name}...") 
                
                res = evaluate_checkpoint(str(ckpt_file), V5_CONFIG, args.bpe_model, test_data, device)
                
                if res:
                    all_results.append(res)
                    print(f"-> WER: {res['avg_wer']:.2f}%")
                    
                    print("\n   MẪU KẾT QUẢ DỰ ĐOÁN:")
                    for i, example in enumerate(res['examples'][:3]): 
                        print(f"   Sample {i+1}:")
                        print(f"     REF: {example['ref']}")
                        print(f"     HYP: {example['hyp']}")
                        print("   " + "-"*40)
    else:
        print(f"Không tìm thấy thư mục checkpoint: {checkpoint_root}")

    if all_results:
        print("\n" + "="*50)
        best = min(all_results, key=lambda x: x["avg_wer"])
        print(f"BEST MODEL: {Path(best['checkpoint']).name}")
        print(f"WER: {best['avg_wer']:.2f}%")
        print("="*50)

run_evaluation()

In [ ]:
import os
import pandas as pd
import torch
import torchaudio
import soundfile as sf
import io
from torch.utils.data import IterableDataset
from tqdm.auto import tqdm
from pathlib import Path
from typing import Dict
import sentencepiece as spm

class ViMDStreamingDataset(IterableDataset):
    def __init__(self, data_dir, num_shards, max_shards=None, max_samples_per_shard=None):
        """
        data_dir: Đường dẫn thư mục chứa file parquet (VD: /kaggle/input/vimd-dataset/test)
        num_shards: Tổng số shard để format tên file (VD: 14 -> 00014)
        """
        self.data_dir = data_dir 
        self.num_shards = num_shards
        self.max_shards = max_shards if max_shards else num_shards
        self.max_samples_per_shard = max_samples_per_shard

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        
        for shard_idx in range(self.max_shards):
            if worker_info is not None and shard_idx % worker_info.num_workers != worker_info.id:
                continue
                
            print(f"Reading Shard {shard_idx + 1}/{self.max_shards} from local...")
            
            try:
                filename = f"test-{shard_idx:05d}-of-{self.num_shards:05d}.parquet"
                local_path = os.path.join(self.data_dir, filename)
                
                if not os.path.exists(local_path):
                    print(f"File not found: {local_path}")
                    continue
                df = pd.read_parquet(local_path)
                
                if "region" in df.columns:
                    original_len = len(df)
                    df = df[df["region"] == "South"]
                else:
                    pass
                count = 0
                for _, row in df.iterrows():
                    if self.max_samples_per_shard and count >= self.max_samples_per_shard:
                        break
                        
                    try:
                        audio_bytes = row["audio"]["bytes"]
                        text = row["text"]
                        
                        audio_array, sr = sf.read(io.BytesIO(audio_bytes))
                        
                        if len(audio_array.shape) > 1:
                            audio_array = audio_array.mean(axis=1)
                        
                        if audio_array.max() > 100: # Fix lỗi scale
                             audio_array = audio_array / 32768.0
                            
                        waveform = torch.tensor(audio_array).float()
                        
                        if sr != 16000:
                            waveform = torchaudio.functional.resample(waveform, sr, 16000)
                        
                        yield {
                            "waveform": waveform,
                            "text": text.upper() if text else ""
                        }
                        count += 1
                        
                    except Exception as e:
                        continue
                        
            except Exception as e:
                print(f"Failed to read shard {shard_idx}: {e}")
                continue

def evaluate_checkpoint(checkpoint_path: str, config: dict, bpe_model: str, 
                        dataset_iterable, device: torch.device) -> Dict:

    
    try:
        model = create_model(config) 
    except Exception as e:
        print(f"Lỗi tạo kiến trúc model: {e}")
        return None

    try:
        ckpt = torch.load(checkpoint_path, map_location='cpu')
        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        else:
            state_dict = ckpt
        model.load_state_dict(state_dict, strict=True)
    except Exception as e:
        print(f"Lỗi load checkpoint: {e}")
        return None

    model = model.to(device).eval()
    sp = spm.SentencePieceProcessor()
    sp.load(bpe_model)
    
    total_errors, total_words = 0, 0
    results = []
    
    for i, sample in enumerate(dataset_iterable):
        try:
            fbank = compute_fbank(sample["waveform"]) 
            hyp = greedy_decode(model, fbank, sp, device)
            ref = sample["text"]
            
            wer, errors, ref_len, _, _ = compute_wer(ref, hyp)
            total_errors += errors
            total_words += ref_len
            
            results.append({"ref": ref, "hyp": hyp, "wer": wer})
        except Exception as e:
            continue
            
    avg_wer = total_errors / total_words * 100 if total_words > 0 else 0
    return { "checkpoint": checkpoint_path, "avg_wer": avg_wer, "examples": results[:3] }

def run_evaluation():
    KAGGLE_DATA_DIR = "/kaggle/input/vimd-dataset/test"
    
    NUM_SHARDS = 14 
    
    print("Khởi tạo ViMD Streaming Dataset (Local Kaggle)...")
    dataset = ViMDStreamingDataset(
        data_dir=KAGGLE_DATA_DIR,      
        num_shards=NUM_SHARDS,         
        max_shards=args.max_shards_to_test, 
        max_samples_per_shard=args.max_samples_per_shard
    )

    print("Đang pre-load dữ liệu vào RAM...")
    test_data = list(tqdm(dataset, desc="Loading Data"))
    
    if not test_data:
        print("Không load được dữ liệu nào! Kiểm tra đường dẫn.")
        return

    print(f"Tổng số mẫu test: {len(test_data)}")
    
    all_results = []
    device = torch.device(args.device)
    checkpoint_root = Path(args.checkpoint_dir) 

    if checkpoint_root.exists():
        print(f"\n--- SCANNING CHECKPOINTS IN {checkpoint_root} ---")
        
        for ckpt_file in sorted(checkpoint_root.glob("*.pt")):
            if "epoch" in ckpt_file.name or ckpt_file.name in ["epoch_3.pt"]:
                print(f"\nEvaluating {ckpt_file.name}...") 
                
                res = evaluate_checkpoint(str(ckpt_file), V5_CONFIG, args.bpe_model, test_data, device)
                
                if res:
                    all_results.append(res)
                    print(f"-> WER: {res['avg_wer']:.2f}%")
                    
                    print("\n   MẪU KẾT QUẢ DỰ ĐOÁN:")
                    for i, example in enumerate(res['examples'][:3]): 
                        print(f"   Sample {i+1}:")
                        print(f"     REF: {example['ref']}")
                        print(f"     HYP: {example['hyp']}")
                        print("   " + "-"*40)
    else:
        print(f"Không tìm thấy thư mục checkpoint: {checkpoint_root}")

    if all_results:
        print("\n" + "="*50)
        best = min(all_results, key=lambda x: x["avg_wer"])
        print(f"BEST MODEL: {Path(best['checkpoint']).name}")
        print(f"WER: {best['avg_wer']:.2f}%")
        print("="*50)

run_evaluation()